# Regime Detection Through Cross-Asset Option-Implied Stress and Event-Conditioned PCA

## Working Paper Draft

This notebook is an end-to-end research paper draft. It studies whether cross-asset option-implied volatility, skew, and PCA concentration can identify stress regimes early enough to improve a simple S&P 500 exposure rule.

The empirical design separates three layers:

1. **Long-history option-implied stress layer:** Nasdaq AR/IVM futures-options data from 2007-current across equities, rates, FX, energy, metals, and agriculture.
2. **Benchmark stress layer:** daily SPX/VIX/rates/DXY data from the existing Bloomberg workbook.
3. **Intraday attribution case study:** the 140-day 60-minute Bloomberg workbook for ES impulse conditional correlation and PCA.


## Abstract

Traditional stress indicators such as VIX, volatility term structure, drawdown state, and yield-curve slopes provide useful but incomplete warnings of equity-market drawdowns. This paper tests whether a cross-asset option-implied layer adds information. The core dataset is a daily futures-options implied-volatility panel covering major equity, rates, FX, energy, metals, and agricultural futures options from 2007 onward. From this panel, the paper constructs constant-tenor measures of ATM implied volatility, risk reversals, butterflies, selected-maturity futures prices, cross-asset correlation, and PCA concentration.

The evidence supports a measured conclusion. Cross-asset option-implied stress features clearly identify periods when future SPX drawdown risk is elevated, especially through equity-index ATM implied volatility and downside-skew pressure. PCA helps compress a high-dimensional futures-options universe into interpretable stress concentration measures. However, simple VIX term-structure benchmarks remain difficult to beat in a chronological holdout. The resulting protection overlay reduces drawdowns and tail exposure, but it often sacrifices upside. The current contribution is therefore best framed as a disciplined stress-regime measurement and de-risking layer, not a stand-alone alpha model.

**Keywords:** cross-asset correlation, implied volatility, futures options, PCA, stress regimes, drawdown protection, risk reversals, S&P 500 timing, point-in-time validation.


## Executive Result

The practical question is whether a portfolio could have avoided some equity drawdown by identifying stress regimes. The notebook therefore begins and ends with a simple comparison: buy and hold SPX total return versus de-risking overlays that reduce exposure after point-in-time stress signals.

The purpose is not to build a final trading product. The purpose is to test whether the cross-asset stress layer is strong enough to deserve more formal strategy research.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root / "src"))

from chronoswan.experiments.intraday_impulse_pca import run_intraday_impulse_pca
from chronoswan.experiments.literature_benchmarks import (
    FEATURE_GROUPS,
    LITERATURE_ANCHORS,
    run_literature_benchmark_replication,
)
from chronoswan.models.logistic import fit_predict_logistic
from chronoswan.validation.metrics import evaluate_probabilities

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

plt.rcParams.update({
    "figure.figsize": (11, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})

reports_dir = project_root / "reports"
reports_dir.mkdir(exist_ok=True)
processed_dir = project_root / "data" / "processed"
raw_dir = project_root / "data" / "raw"

daily_path = raw_dir / "Book1.xlsx"
intraday_path = project_root / "data" / "17sheets.xlsx"
ar_feature_path = processed_dir / "ar_ivm_extended_constant_tenor_features_2007_current.parquet"
ar_coverage_path = processed_dir / "ar_ivm_extended_coverage_2007_current.csv"
ar_raw_path = raw_dir / "ar_ivm_extended_2007_current.parquet"

if not ar_feature_path.exists():
    ar_feature_path = processed_dir / "ar_ivm_constant_tenor_features_2007_current.parquet"
    ar_coverage_path = processed_dir / "ar_ivm_coverage_2007_current.csv"
    ar_raw_path = raw_dir / "ar_ivm_2007_current.parquet"

paths = pd.DataFrame([
    {"dataset": "Daily Bloomberg stress panel", "path": daily_path.relative_to(project_root), "exists": daily_path.exists()},
    {"dataset": "Nasdaq AR/IVM feature panel", "path": ar_feature_path.relative_to(project_root), "exists": ar_feature_path.exists()},
    {"dataset": "Nasdaq AR/IVM raw pull", "path": ar_raw_path.relative_to(project_root), "exists": ar_raw_path.exists()},
    {"dataset": "Intraday Bloomberg ES workbook", "path": intraday_path.relative_to(project_root), "exists": intraday_path.exists()},
])
paths


The notebook uses local data files only. Raw vendor data, derived parquet files, and executed reports remain ignored by git.


In [ ]:
daily_result = run_literature_benchmark_replication(daily_path, output_dir=processed_dir)
daily_panel = daily_result["panel"].copy()
conditional_rates = daily_result["conditional_rates"]
daily_model_results = daily_result["model_results"]
daily_case_table = daily_result["case_table"]

ar_features = pd.read_parquet(ar_feature_path)
ar_coverage = pd.read_csv(ar_coverage_path, parse_dates=["start", "end"])

intraday_result = run_intraday_impulse_pca(intraday_path, output_dir=processed_dir)
coverage = intraday_result["coverage"]
event_summary = intraday_result["event_summary"]
conditional_correlations = intraday_result["conditional_correlations"]
pca_summary = intraday_result["pca_summary"]
pca_loadings = intraday_result["pca_loadings"]
rolling_pca = intraday_result["rolling_pca"]
predictive_results = intraday_result["predictive_results"]

pd.DataFrame([
    {
        "block": "daily benchmark",
        "rows": len(daily_panel),
        "start": daily_panel["date"].min(),
        "end": daily_panel["date"].max(),
        "columns": daily_panel.shape[1],
    },
    {
        "block": "AR/IVM option-implied features",
        "rows": len(ar_features),
        "start": ar_features.index.min(),
        "end": ar_features.index.max(),
        "columns": ar_features.shape[1],
    },
    {
        "block": "intraday ES impulse workbook",
        "rows": len(intraday_result["long_frame"]),
        "start": intraday_result["long_frame"]["timestamp"].min(),
        "end": intraday_result["long_frame"]["timestamp"].max(),
        "columns": coverage["ticker"].nunique(),
    },
])


The long-history options panel is the empirical backbone for the stress-regime study. The intraday workbook is retained as an attribution case study because its 140-day window is too short for cross-regime validation.


## 1. Motivation: Drawdowns And Compounding

A stress-regime detector does not need to predict every market move. It can be useful if it avoids a portion of major drawdowns without giving up too much compounding. This section documents the benchmark problem using SPX total return.


In [ ]:
def max_drawdown_series(equity: pd.Series) -> pd.Series:
    return equity / equity.cummax() - 1.0

spxt = daily_panel.dropna(subset=["SPXT"]).copy()
spxt["spxt_return"] = spxt["SPXT"].pct_change().fillna(0.0)
spxt["spxt_growth"] = (1 + spxt["spxt_return"]).cumprod()
spxt["drawdown"] = max_drawdown_series(spxt["spxt_growth"])

largest_drawdowns = spxt.nsmallest(10, "drawdown")[["date", "SPX", "SPXT", "drawdown", "VIX", "vix3m_minus_vix", "realized_vol_20"]]
largest_drawdowns


The largest drawdown observations cluster around recognizable stress regimes. This motivates a protection layer, but it also raises the standard: the rule must be point-in-time and must not simply identify crises after they have happened.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
axes[0].plot(spxt["date"], spxt["spxt_growth"], color="#1f6f8b", linewidth=1.4)
axes[0].set_title("SPX total return growth of $1")
axes[0].set_ylabel("Growth of $1")
axes[1].fill_between(spxt["date"], spxt["drawdown"] * 100, 0, color="#9b3d2e", alpha=0.55)
axes[1].set_title("SPX total return drawdown")
axes[1].set_ylabel("Drawdown (%)")
fig.tight_layout()
fig.savefig(reports_dir / "paper_spxt_motivation_drawdown.png", dpi=160)
plt.show()


The compounding problem is visible. A regime detector is valuable only if it cuts exposure before or during the damaging part of these drawdowns while avoiding excessive time out of the market.


## 2. Literature And Benchmark Positioning

The research is adjacent to well-established work on tail dependence, contagion, dynamic conditional correlation, spillovers, option-implied tail risk, and PCA absorption ratios. The contribution is not that PCA or implied volatility are new. The contribution is a disciplined cross-asset stress pipeline with explicit benchmarks and de-risking tests.


In [ ]:
literature = pd.DataFrame(LITERATURE_ANCHORS)
literature


The benchmark map starts from standard stress measures. A cross-asset options layer must improve on, or at least clarify, VIX and volatility term-structure signals.


## 3. Data: Futures Options Implied Volatility

The Nasdaq AR/IVM feed provides daily implied-volatility model values for futures options. The expanded pull includes equity index, rates, FX, energy, metals, and agricultural futures options. For each root, the pipeline selects nearest 30d, 60d, and 90d tenors and keeps selected-maturity futures price, ATM implied volatility, risk reversals, and butterflies.


In [ ]:
ar_coverage[["research_ticker", "rows", "start", "end", "unique_dates", "min_dte", "median_dte", "max_dte"]]


The expanded universe includes grains and oilseeds in addition to the macro/risk roots. This is useful because food and energy inflation shocks can be part of macro stress regimes, while PCA keeps the larger panel from becoming unmanageable.


In [ ]:
def infer_roots(columns: pd.Index) -> list[str]:
    return sorted({column.split("_")[0] for column in columns})

ROOT_GROUPS = {
    "equity": ["ES", "NQ", "RTY"],
    "rates": ["TU", "FV", "TY", "TN", "US", "UB"],
    "fx": ["DX", "AD", "BP", "CD", "EC", "JY", "SF"],
    "energy": ["CL", "B", "HO", "NG", "RB"],
    "metals": ["GC", "HG", "SI"],
    "agriculture": ["BO", "C", "S", "SM", "W"],
}

roots = infer_roots(ar_features.columns)
root_map = []
for group, group_roots in ROOT_GROUPS.items():
    for root in group_roots:
        root_map.append({"root": root, "group": group, "available": root in roots})
root_map = pd.DataFrame(root_map)
root_map.pivot_table(index="group", values="available", aggfunc=["sum", "count"])


The universe is broad enough for cross-asset stress work. Some roots start later than 2007, so all PCA and model sections use availability checks rather than assuming a balanced panel from inception.


In [ ]:
preview_columns = [
    column for column in [
        "ES_futures_30d", "ES_atm_30d", "ES_rr25_30d", "ES_fly25_30d",
        "TY_atm_30d", "CL_atm_30d", "GC_atm_30d", "C_atm_30d", "W_atm_30d",
    ]
    if column in ar_features.columns
]
ar_features[preview_columns].dropna(how="all").head(5)


The feature panel is already in model-ready daily form. The raw AR/IVM table is much larger because it contains many expiries per root and date.


## 4. Cross-Asset Correlation Structure

This section studies two related objects: correlations of selected-maturity futures returns and correlations of 30d ATM implied-volatility changes. The former approximates realized cross-asset price co-movement; the latter captures option-implied stress co-movement.


In [ ]:
def build_root_panel(features: pd.DataFrame, field: str, tenor: int = 30) -> pd.DataFrame:
    selected = {
        root: features[f"{root}_{field}_{tenor}d"]
        for root in infer_roots(features.columns)
        if f"{root}_{field}_{tenor}d" in features.columns
    }
    return pd.DataFrame(selected, index=features.index).sort_index()

atm30 = build_root_panel(ar_features, "atm", 30)
rr25_30 = build_root_panel(ar_features, "rr25", 30)
futures30 = build_root_panel(ar_features, "futures", 30)

atm_change = atm30.diff()
futures_return = np.log(futures30.where(futures30 > 0)).diff()

post_start = "2018-01-01"
atm_corr_es = (
    atm_change.loc[post_start:]
    .dropna(axis=1, thresh=1000)
    .corr()["ES"]
    .sort_values(ascending=False)
    .rename("corr_with_ES_30d_ATM_vol_change")
    .reset_index()
    .rename(columns={"index": "root"})
)
futures_corr_es = (
    futures_return.loc[post_start:]
    .dropna(axis=1, thresh=1000)
    .corr()["ES"]
    .sort_values(ascending=False)
    .rename("corr_with_ES_selected_futures_return")
    .reset_index()
    .rename(columns={"index": "root"})
)
display(atm_corr_es)
display(futures_corr_es)


The correlations are intuitive. ES implied-vol changes co-move strongly with other equity-index vol changes and moderately with rates, FX, metals, and some commodities. Selected futures-price returns recover the usual positive equity beta and negative dollar/rates relationships.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_atm = atm_corr_es.sort_values("corr_with_ES_30d_ATM_vol_change")
axes[0].barh(plot_atm["root"], plot_atm["corr_with_ES_30d_ATM_vol_change"], color="#1f6f8b")
axes[0].axvline(0, color="#333333", linewidth=1)
axes[0].set_title("30d ATM-vol change correlation with ES")
axes[0].set_xlabel("Correlation")
plot_fut = futures_corr_es.sort_values("corr_with_ES_selected_futures_return")
colors = np.where(plot_fut["corr_with_ES_selected_futures_return"] >= 0, "#1f6f8b", "#9b3d2e")
axes[1].barh(plot_fut["root"], plot_fut["corr_with_ES_selected_futures_return"], color=colors)
axes[1].axvline(0, color="#333333", linewidth=1)
axes[1].set_title("Selected futures-return correlation with ES")
axes[1].set_xlabel("Correlation")
fig.tight_layout()
fig.savefig(reports_dir / "paper_ar_ivm_cross_asset_correlations.png", dpi=160)
plt.show()


Including agriculture does not dilute the analysis if the variables are grouped and summarized. Agricultural implied volatility is not expected to behave like equity beta, but it can help identify inflation/supply-shock regimes.


## 5. PCA Stress Concentration

PCA is useful here because the expanded panel is intentionally broad. Rather than manually interpret dozens of implied-volatility roots, the notebook estimates whether cross-asset option-implied moves are becoming more concentrated in a small number of components.


In [ ]:
def rolling_pca_absorption(frame: pd.DataFrame, *, window: int = 252, step: int = 5, n_components: int = 3, min_rows: int = 160, min_cols: int = 6) -> pd.DataFrame:
    rows = []
    for end in range(window, len(frame), step):
        win = frame.iloc[end - window:end]
        cols = [column for column in win.columns if win[column].notna().mean() >= 0.80]
        clean = win[cols].dropna()
        if len(cols) < min_cols or len(clean) < min_rows:
            absorption = np.nan
        else:
            x = StandardScaler().fit_transform(clean)
            pca = PCA(n_components=min(n_components, x.shape[1], x.shape[0] - 1), random_state=7)
            absorption = float(pca.fit(x).explained_variance_ratio_.sum())
        rows.append({"date": frame.index[end - 1], "absorption_3pc": absorption, "n_roots": len(cols), "n_rows": len(clean)})
    return pd.DataFrame(rows).set_index("date")

atm_absorption = rolling_pca_absorption(atm_change)
futures_absorption = rolling_pca_absorption(futures_return)
absorption_summary = pd.DataFrame([
    {"panel": "ATM implied-vol changes", **atm_absorption["absorption_3pc"].describe().to_dict()},
    {"panel": "selected futures returns", **futures_absorption["absorption_3pc"].describe().to_dict()},
])
absorption_summary


The options-implied panel has a meaningful concentration cycle. High absorption means a larger share of cross-asset implied-volatility movement is being driven by a small number of common factors.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(atm_absorption.index, atm_absorption["absorption_3pc"] * 100, label="ATM vol changes", color="#1f6f8b")
ax.plot(futures_absorption.index, futures_absorption["absorption_3pc"] * 100, label="selected futures returns", color="#9a6a20", alpha=0.8)
ax.set_title("Rolling cross-asset PCA absorption")
ax.set_ylabel("First 3 PCs variance share (%)")
ax.legend()
fig.tight_layout()
fig.savefig(reports_dir / "paper_ar_ivm_pca_absorption.png", dpi=160)
plt.show()


PCA is a dimension-reduction and regime-measurement tool here. It should not be treated as causal attribution by itself.


## 6. Constructing The Option-Implied Stress Layer

The stress layer combines point-in-time z-scores from equity ATM volatility, cross-asset average ATM volatility, cross-sectional dispersion, ES downside-skew pressure, and PCA absorption. Group-level features are retained so agriculture, energy, rates, FX, and metals can contribute without overwhelming the model.


In [ ]:
def rolling_zscore(series: pd.Series, window: int = 252, min_periods: int = 126) -> pd.Series:
    mean = series.rolling(window, min_periods=min_periods).mean().shift(1)
    std = series.rolling(window, min_periods=min_periods).std().shift(1)
    return (series - mean) / std.replace(0, np.nan)


def group_mean(panel: pd.DataFrame, roots_for_group: list[str]) -> pd.Series:
    cols = [root for root in roots_for_group if root in panel.columns]
    if not cols:
        return pd.Series(np.nan, index=panel.index)
    return panel[cols].mean(axis=1)

stress = pd.DataFrame(index=ar_features.index)
stress["ar_mean_atm_30d"] = atm30.mean(axis=1)
stress["ar_atm_dispersion_30d"] = atm30.std(axis=1)
stress["ar_es_atm_30d"] = atm30.get("ES")
stress["ar_es_rr25_30d"] = rr25_30.get("ES")
for group, group_roots in ROOT_GROUPS.items():
    stress[f"ar_{group}_atm_30d"] = group_mean(atm30, group_roots)

stress = stress.join(atm_absorption[["absorption_3pc"]].rename(columns={"absorption_3pc": "ar_atm_absorption_3pc"}), how="left")
stress["ar_atm_absorption_3pc"] = stress["ar_atm_absorption_3pc"].ffill(limit=5)

for column in list(stress.columns):
    stress[f"{column}_z"] = rolling_zscore(stress[column])

stress_components = pd.DataFrame(index=stress.index)
stress_components["equity_vol"] = stress["ar_equity_atm_30d_z"]
stress_components["broad_vol"] = stress["ar_mean_atm_30d_z"]
stress_components["dispersion"] = stress["ar_atm_dispersion_30d_z"]
stress_components["es_skew_pressure"] = -stress["ar_es_rr25_30d_z"]
stress_components["pca_absorption"] = stress["ar_atm_absorption_3pc_z"]
stress["ar_stress_score"] = stress_components.mean(axis=1)
stress["ar_stress_score_z"] = rolling_zscore(stress["ar_stress_score"])

stress_preview = stress[[
    "ar_stress_score", "ar_stress_score_z", "ar_equity_atm_30d_z", "ar_energy_atm_30d_z",
    "ar_agriculture_atm_30d_z", "ar_es_rr25_30d_z", "ar_atm_absorption_3pc_z",
]].dropna().tail(10)
stress_preview


The stress score is built only from values known at each date, using shifted rolling z-scores. More negative ES risk reversal increases the stress score through the skew-pressure component.


In [ ]:
analysis_panel = daily_panel.merge(stress.reset_index().rename(columns={"index": "date"}), on="date", how="left")
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(analysis_panel["date"], analysis_panel["ar_stress_score_z"], color="#1f6f8b", linewidth=1.1, label="AR/IVM stress score")
ax1.axhline(1.5, color="#333333", linewidth=1, linestyle="--")
ax1.set_ylabel("Stress score z")
ax1.set_title("Option-implied cross-asset stress score")
ax2 = ax1.twinx()
ax2.plot(analysis_panel["date"], analysis_panel["drawdown_252"] * 100, color="#9b3d2e", alpha=0.45, linewidth=1.0, label="SPX drawdown")
ax2.set_ylabel("SPX 252D drawdown (%)")
fig.tight_layout()
fig.savefig(reports_dir / "paper_ar_ivm_stress_score.png", dpi=160)
plt.show()


The stress score rises around major volatility regimes, but it is not a perfect switch. The model must therefore be judged by event-rate lift and backtest behavior, not visual fit alone.


## 7. Conditional Event-Rate Tests

The first empirical test is simple: when option-implied stress features are extreme, does the forward SPX drawdown event rate rise relative to the base rate?


In [ ]:
def conditional_lift_table(panel: pd.DataFrame, specs: dict[str, tuple[str, str]], targets: tuple[str, ...] = ("event_dd5_h20", "event_dd10_h20")) -> pd.DataFrame:
    rows = []
    for label, (column, direction) in specs.items():
        if column not in panel:
            continue
        valid = panel[column].notna()
        if valid.sum() == 0:
            continue
        threshold = panel.loc[valid, column].quantile(0.90 if direction == "high" else 0.10)
        mask = valid & ((panel[column] >= threshold) if direction == "high" else (panel[column] <= threshold))
        for target in targets:
            base = panel[target].mean()
            conditional = panel.loc[mask, target].mean()
            rows.append({
                "indicator": label,
                "target": target,
                "coverage": mask.mean(),
                "base_event_rate": base,
                "conditional_event_rate": conditional,
                "lift_vs_base": conditional / base if base else np.nan,
            })
    return pd.DataFrame(rows).sort_values(["target", "lift_vs_base"], ascending=[True, False])

indicator_specs = {
    "AR stress score top decile": ("ar_stress_score_z", "high"),
    "AR equity ATM top decile": ("ar_equity_atm_30d_z", "high"),
    "AR broad ATM top decile": ("ar_mean_atm_30d_z", "high"),
    "AR ES RR25 bottom decile": ("ar_es_rr25_30d_z", "low"),
    "AR energy ATM top decile": ("ar_energy_atm_30d_z", "high"),
    "AR agriculture ATM top decile": ("ar_agriculture_atm_30d_z", "high"),
    "AR PCA absorption top decile": ("ar_atm_absorption_3pc_z", "high"),
}
option_lifts = conditional_lift_table(analysis_panel, indicator_specs)
option_lifts.assign(
    base_event_rate_pct=lambda x: x["base_event_rate"] * 100,
    conditional_event_rate_pct=lambda x: x["conditional_event_rate"] * 100,
)[["indicator", "target", "coverage", "base_event_rate_pct", "conditional_event_rate_pct", "lift_vs_base"]]


The option-implied layer has event-rate signal. Equity-index ATM volatility and downside-skew pressure are the clearest stress indicators. Agriculture and energy features are more regime-specific; they are useful context rather than universal equity-crash predictors.


In [ ]:
plot_lifts = option_lifts.loc[option_lifts["target"].eq("event_dd5_h20")].copy().sort_values("lift_vs_base")
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.barh(plot_lifts["indicator"], plot_lifts["lift_vs_base"], color="#1f6f8b")
ax.axvline(1.0, color="#333333", linewidth=1)
ax.set_title("Forward 5% SPX drawdown event-rate lift")
ax.set_xlabel("Conditional event rate / base event rate")
fig.tight_layout()
fig.savefig(reports_dir / "paper_option_indicator_lifts.png", dpi=160)
plt.show()


The lift table justifies keeping the option-implied stress layer. It does not yet prove that a strategy will outperform after costs and signal delay.


## 8. Forecasting Benchmark

The next test compares feature groups in a chronological holdout. Models are trained before 2020 and tested from 2020 onward. This is intentionally difficult because the holdout includes COVID, inflation/rates stress, and later volatility shocks.


In [ ]:
def run_holdout_feature_models(panel: pd.DataFrame, target: str = "event_dd5_h20") -> tuple[pd.DataFrame, dict[str, pd.Series], pd.DataFrame, pd.DataFrame]:
    model_frame = panel.dropna(subset=[target]).reset_index(drop=True)
    train_index = model_frame.index[model_frame["date"] < "2020-01-01"].to_numpy()
    if len(train_index) > 20:
        train_index = train_index[:-20]
    test_index = model_frame.index[model_frame["date"] >= "2020-01-01"].to_numpy()
    train = model_frame.iloc[train_index].copy()
    test = model_frame.iloc[test_index].copy()
    ar_candidates = [
        "ar_stress_score_z",
        "ar_equity_atm_30d_z",
        "ar_rates_atm_30d_z",
        "ar_fx_atm_30d_z",
        "ar_energy_atm_30d_z",
        "ar_metals_atm_30d_z",
        "ar_agriculture_atm_30d_z",
        "ar_atm_dispersion_30d_z",
        "ar_es_rr25_30d_z",
        "ar_atm_absorption_3pc_z",
    ]
    groups = {
        "base_rate": [],
        "vix_term_structure": FEATURE_GROUPS["vix_term_structure"],
        "option_implied_stress": ar_candidates,
        "vix_plus_option_stress": FEATURE_GROUPS["vix_term_structure"] + ar_candidates,
    }
    rows = []
    probabilities = {}
    for group_name, candidates in groups.items():
        features = [
            feature for feature in candidates
            if feature in model_frame and train[feature].notna().mean() > 0.70 and test[feature].notna().mean() > 0.70
        ]
        if not features:
            probs = np.repeat(train[target].mean(), len(test))
            train_probs = np.repeat(train[target].mean(), len(train))
        else:
            fit = fit_predict_logistic(train[features], train[target], test[features], class_weight=None, random_state=7)
            probs = fit.probabilities
            train_probs = fit.model.predict_proba(train[features])[:, 1]
        rows.append({"feature_group": group_name, "n_features": len(features), **evaluate_probabilities(test[target], probs)})
        probabilities[group_name] = pd.Series(probs, index=test.index)
        probabilities[f"{group_name}_train"] = pd.Series(train_probs, index=train.index)
    return pd.DataFrame(rows).sort_values("average_precision", ascending=False), probabilities, train, test

holdout_results, holdout_probabilities, holdout_train, holdout_test = run_holdout_feature_models(analysis_panel)
holdout_results


The benchmark remains sobering. VIX term structure is hard to beat. The options-implied stress layer is informative in conditional tables, but its first regularized logistic form is not yet a superior rare-event forecast.


In [ ]:
plot_models = holdout_results.sort_values("average_precision")
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].barh(plot_models["feature_group"], plot_models["average_precision"], color="#1f6f8b")
axes[0].axvline(plot_models["positive_rate"].iloc[0], color="#333333", linestyle="--", linewidth=1)
axes[0].set_title("Holdout average precision")
axes[0].set_xlabel("Average precision")
axes[1].barh(plot_models["feature_group"], plot_models["brier_score"], color="#9a6a20")
axes[1].set_title("Holdout Brier score")
axes[1].set_xlabel("Lower is better")
fig.tight_layout()
fig.savefig(reports_dir / "paper_holdout_model_results.png", dpi=160)
plt.show()


This is a useful result, not a failure. It says the paper should emphasize robust stress measurement and drawdown control before claiming prediction.


## 9. Strategy Design And Backtest

The strategy is a simple exposure overlay on SPX total return. It does not short the market and does not use options directly. When stress is high, exposure is reduced; otherwise exposure remains 100%. Signals are shifted by one day before returns are applied. A 1 bp transaction-cost assumption is charged per unit of exposure turnover.

Three tests are reported:

1. **Holdout model overlay:** train thresholds before 2020, test from 2020 onward.
2. **AR/IVM sensitivity study:** vary the stress threshold and low-risk exposure in the 2020+ holdout.
3. **Expanding stress-score overlay:** full-sample point-in-time illustration using an expanding historical threshold after an initial warm-up.


In [ ]:
def strategy_metrics(returns: pd.Series, position: pd.Series | None = None) -> dict[str, float]:
    clean = pd.Series(returns).dropna()
    equity = (1 + clean).cumprod()
    drawdown = equity / equity.cummax() - 1.0
    annual_return = equity.iloc[-1] ** (252 / len(clean)) - 1 if len(clean) else np.nan
    annual_vol = clean.std() * np.sqrt(252)
    return {
        "n_days": float(len(clean)),
        "cumulative_return": float(equity.iloc[-1] - 1),
        "annual_return": float(annual_return),
        "annual_vol": float(annual_vol),
        "sharpe_zero_rf": float(annual_return / annual_vol) if annual_vol else np.nan,
        "max_drawdown": float(drawdown.min()),
        "worst_day": float(clean.min()),
        "mean_exposure": float(position.mean()) if position is not None else 1.0,
        "turnover": float(position.diff().abs().sum()) if position is not None else 0.0,
    }


def overlay_returns(base_returns: pd.Series, high_risk: pd.Series, *, low_exposure: float = 0.5, cost_per_turnover: float = 0.0001) -> tuple[pd.Series, pd.Series]:
    high_risk = pd.Series(high_risk, index=base_returns.index).fillna(False)
    position = pd.Series(np.where(high_risk, low_exposure, 1.0), index=base_returns.index).shift(1).fillna(1.0)
    costs = position.diff().abs().fillna(0.0) * cost_per_turnover
    return position * base_returns - costs, position

holdout_bt = holdout_test.copy()
holdout_bt["spxt_return"] = holdout_bt["SPXT"].pct_change().fillna(0.0)

vix_train_probs = holdout_probabilities["vix_term_structure_train"]
vix_test_probs = holdout_probabilities["vix_term_structure"]
combined_train_probs = holdout_probabilities["vix_plus_option_stress_train"]
combined_test_probs = holdout_probabilities["vix_plus_option_stress"]

vix_threshold = float(vix_train_probs.quantile(0.85))
combined_threshold = float(combined_train_probs.quantile(0.85))
ar_threshold = float(holdout_train["ar_stress_score_z"].dropna().quantile(0.85))

holdout_bt["vix_high_risk"] = vix_test_probs.reindex(holdout_bt.index) >= vix_threshold
holdout_bt["ar_high_risk"] = holdout_bt["ar_stress_score_z"] >= ar_threshold
holdout_bt["combined_high_risk"] = combined_test_probs.reindex(holdout_bt.index) >= combined_threshold

strategy_records = [{"strategy": "SPXT buy and hold", **strategy_metrics(holdout_bt["spxt_return"])}]
strategy_curves = pd.DataFrame({"date": holdout_bt["date"], "SPXT buy and hold": (1 + holdout_bt["spxt_return"]).cumprod()})
for name, signal in [
    ("VIX-term model overlay", holdout_bt["vix_high_risk"]),
    ("AR/IVM stress overlay", holdout_bt["ar_high_risk"]),
    ("Combined model overlay", holdout_bt["combined_high_risk"]),
]:
    returns, position = overlay_returns(holdout_bt["spxt_return"], signal)
    strategy_records.append({"strategy": name, **strategy_metrics(returns, position)})
    strategy_curves[name] = (1 + returns).cumprod()

holdout_strategy_table = pd.DataFrame(strategy_records)
holdout_strategy_table


The holdout backtest shows the central trade-off. The AR/IVM stress overlay keeps most of the upside while cutting realized volatility, max drawdown, and the worst single day versus buy-and-hold. The combined model does not dominate here, which argues for treating VIX and AR/IVM as separate diagnostics before forcing them into one probability score.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for column in strategy_curves.columns:
    if column != "date":
        ax.plot(strategy_curves["date"], strategy_curves[column], label=column, linewidth=1.4)
ax.set_title("2020+ holdout: SPX total return versus de-risking overlays")
ax.set_ylabel("Growth of $1")
ax.legend()
fig.tight_layout()
fig.savefig(reports_dir / "paper_holdout_strategy_curves.png", dpi=160)
plt.show()


The strategy result should be read as a risk-control result. The baseline AR/IVM overlay improves risk-adjusted behavior in the 2020+ holdout, while buy-and-hold remains the clean raw-return benchmark.

In [ ]:
ar_quantiles = [0.80, 0.85, 0.90, 0.95]
low_exposure_grid = [0.00, 0.25, 0.50, 0.75]

sensitivity_records = []
sensitivity_curves = pd.DataFrame({
    "date": holdout_bt["date"],
    "SPXT buy and hold": (1 + holdout_bt["spxt_return"]).cumprod(),
})

for quantile in ar_quantiles:
    threshold = float(holdout_train["ar_stress_score_z"].dropna().quantile(quantile))
    high_risk = holdout_bt["ar_stress_score_z"] >= threshold
    for low_exposure in low_exposure_grid:
        returns, position = overlay_returns(holdout_bt["spxt_return"], high_risk, low_exposure=low_exposure)
        metrics = strategy_metrics(returns, position)
        sensitivity_records.append({
            "stress_quantile": quantile,
            "low_exposure": low_exposure,
            "threshold": threshold,
            "high_risk_days_pct": float(high_risk.mean() * 100),
            **metrics,
        })
        if quantile == 0.95 and low_exposure == 0.00:
            sensitivity_curves["AR/IVM 95th pct cash overlay"] = (1 + returns).cumprod()
        if quantile == 0.85 and low_exposure == 0.50:
            sensitivity_curves["AR/IVM 85th pct 50% overlay"] = (1 + returns).cumprod()

sensitivity_table = pd.DataFrame(sensitivity_records)
sensitivity_display = sensitivity_table.sort_values("sharpe_zero_rf", ascending=False).head(10)[[
    "stress_quantile", "low_exposure", "high_risk_days_pct", "cumulative_return", "annual_return",
    "annual_vol", "sharpe_zero_rf", "max_drawdown", "worst_day", "turnover",
]]
sensitivity_display

The high-conviction AR/IVM rule is the interesting sensitivity result. In this holdout, waiting for the most extreme stress-score states reduces the time spent de-risked and can improve both drawdown control and compounding, but this must be treated as a candidate rule to lock before any further testing.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for low_exposure in low_exposure_grid:
    subset = sensitivity_table.loc[sensitivity_table["low_exposure"].eq(low_exposure)].sort_values("stress_quantile")
    axes[0].plot(subset["stress_quantile"], subset["sharpe_zero_rf"], marker="o", label=f"low exposure {low_exposure:.0%}")
    axes[1].plot(subset["stress_quantile"], subset["max_drawdown"] * 100, marker="o", label=f"low exposure {low_exposure:.0%}")
axes[0].axhline(holdout_strategy_table.loc[holdout_strategy_table["strategy"].eq("SPXT buy and hold"), "sharpe_zero_rf"].iloc[0], color="#333333", linestyle="--", linewidth=1)
axes[0].set_title("AR/IVM holdout sensitivity: Sharpe")
axes[0].set_xlabel("Pre-2020 stress-score threshold quantile")
axes[0].set_ylabel("Sharpe, zero rf")
axes[1].axhline(holdout_strategy_table.loc[holdout_strategy_table["strategy"].eq("SPXT buy and hold"), "max_drawdown"].iloc[0] * 100, color="#333333", linestyle="--", linewidth=1)
axes[1].set_title("AR/IVM holdout sensitivity: max drawdown")
axes[1].set_xlabel("Pre-2020 stress-score threshold quantile")
axes[1].set_ylabel("Max drawdown (%)")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(reports_dir / "paper_ar_ivm_strategy_sensitivity.png", dpi=160)
plt.show()

The sensitivity curve favors stricter thresholds over constant caution. That is economically plausible: the option-implied layer is strongest when it identifies unusually concentrated stress, not when it tries to micromanage every mildly elevated volatility state.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for column in sensitivity_curves.columns:
    if column != "date":
        linewidth = 1.8 if column == "AR/IVM 95th pct cash overlay" else 1.2
        ax.plot(sensitivity_curves["date"], sensitivity_curves[column], label=column, linewidth=linewidth)
ax.set_title("2020+ holdout: high-conviction AR/IVM stress overlay")
ax.set_ylabel("Growth of $1")
ax.legend()
fig.tight_layout()
fig.savefig(reports_dir / "paper_ar_ivm_high_conviction_curve.png", dpi=160)
plt.show()

The high-conviction curve is not the final strategy claim, but it is a useful research lead. It suggests the AR/IVM stress layer may be most valuable as an intermittent de-risking trigger rather than a continuous market-timing score.

In [ ]:
full_bt = analysis_panel.dropna(subset=["SPXT", "ar_stress_score_z"]).copy()
full_bt["spxt_return"] = full_bt["SPXT"].pct_change().fillna(0.0)
expanding_threshold = full_bt["ar_stress_score_z"].expanding(min_periods=756).quantile(0.85).shift(1)
full_bt["expanding_ar_high_risk"] = full_bt["ar_stress_score_z"] >= expanding_threshold
full_overlay_returns, full_position = overlay_returns(full_bt["spxt_return"], full_bt["expanding_ar_high_risk"])
full_strategy = pd.DataFrame([
    {"strategy": "SPXT buy and hold", **strategy_metrics(full_bt["spxt_return"])},
    {"strategy": "Expanding AR/IVM stress overlay", **strategy_metrics(full_overlay_returns, full_position)},
])
full_curves = pd.DataFrame({
    "date": full_bt["date"],
    "SPXT buy and hold": (1 + full_bt["spxt_return"]).cumprod(),
    "Expanding AR/IVM stress overlay": (1 + full_overlay_returns).cumprod(),
    "position": full_position,
})
full_strategy


The expanding-threshold test is a point-in-time illustration over the longer sample. It is not optimized by crisis window; the threshold uses only prior stress-score history.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
axes[0].plot(full_curves["date"], full_curves["SPXT buy and hold"], label="SPXT buy and hold", color="#333333")
axes[0].plot(full_curves["date"], full_curves["Expanding AR/IVM stress overlay"], label="Expanding AR/IVM stress overlay", color="#1f6f8b")
axes[0].set_title("Longer-sample point-in-time AR/IVM de-risking illustration")
axes[0].set_ylabel("Growth of $1")
axes[0].legend()
axes[1].plot(full_curves["date"], full_curves["position"], color="#9a6a20", linewidth=1.0)
axes[1].set_ylabel("Exposure")
axes[1].set_ylim(0.45, 1.05)
fig.tight_layout()
fig.savefig(reports_dir / "paper_expanding_ar_strategy_curves.png", dpi=160)
plt.show()


The longer-sample illustration is useful because it makes the compounding trade-off visible across multiple regimes. The AR/IVM overlay lowers volatility and drawdown but gives up some cumulative return, so the next research step is hedge design rather than aggressive threshold tuning.

## 10. Intraday ES Impulse Case Study

The intraday workbook is still valuable because it directly matches the original 60-minute ES impulse idea. It is not long enough to carry the final paper, but it demonstrates how the option-implied stress layer could be paired with event-conditioned realized cross-asset attribution.


In [ ]:
event_summary.assign(event_rate_pct=lambda x: x["event_rate"] * 100)[[
    "event", "eligible_bars", "event_count", "event_rate_pct", "median_es_return_bp", "mean_abs_es_return_bp", "rolling_window_bars",
]]


The intraday event rule isolates meaningful ES bars using a shifted rolling threshold. This remains a clean point-in-time labeling method.


In [ ]:
large_down_corr = conditional_correlations.loc[conditional_correlations["sample"].eq("large_down")].copy()
pc1_summary = pca_summary.loc[pca_summary["component"].eq(1)].copy()
display(large_down_corr.head(10)[["sample", "driver", "n_obs", "corr_with_es", "delta_vs_ready_corr"]])
display(pc1_summary[["sample", "n_rows", "n_drivers", "explained_variance_ratio", "abs_corr_with_es", "top_positive_loadings", "top_negative_loadings"]])


The intraday case study agrees with the broader stress-regime intuition. Large ES moves are more factor-concentrated, and conditional correlations recover a recognizable risk-off basket, but the 140-day sample should be used as a workflow demonstration rather than final evidence.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_corr = large_down_corr.dropna(subset=["corr_with_es"]).head(12).sort_values("corr_with_es")
axes[0].barh(plot_corr["driver"], plot_corr["corr_with_es"], color=np.where(plot_corr["corr_with_es"] >= 0, "#1f6f8b", "#9b3d2e"))
axes[0].axvline(0, color="#333333", linewidth=1)
axes[0].set_title("Large-down ES impulse correlations")
axes[0].set_xlabel("Correlation with ES")
plot_pc1 = pc1_summary.set_index("sample").loc[["threshold_ready", "large_abs", "large_down", "large_up"]].reset_index()
axes[1].bar(plot_pc1["sample"], plot_pc1["explained_variance_ratio"] * 100, color=["#7f8b99", "#1f6f8b", "#9b3d2e", "#3a7f5f"])
axes[1].set_title("Intraday PC1 variance share")
axes[1].set_ylabel("Explained variance (%)")
fig.tight_layout()
fig.savefig(reports_dir / "paper_intraday_case_study.png", dpi=160)
plt.show()


The intraday section should be presented as a bridge to future work: the same framework can attribute realized ES impulses after the options-implied stress layer has identified a high-risk regime.


In [ ]:
final_scorecard = pd.concat([
    holdout_strategy_table.assign(sample="2020+ holdout"),
    full_strategy.assign(sample="2010-current expanding"),
], ignore_index=True)
final_scorecard[[
    "sample", "strategy", "cumulative_return", "annual_return", "annual_vol",
    "sharpe_zero_rf", "max_drawdown", "worst_day", "mean_exposure", "turnover",
]]

The backtest evidence is consistent rather than magical. The stress overlay improves drawdown and volatility metrics, while buy-and-hold remains the benchmark for raw market beta compounding.

## 11. Conclusions

The strongest conclusion is that cross-asset option-implied data is a legitimate stress-regime layer. It is not weaker than realized returns for this research question; it is arguably better suited to the question because it measures the price of uncertainty, downside skew, and convexity demand before realized outcomes are fully visible.

The main empirical relationships are clear:

- ES implied-volatility shocks co-move most with NQ and RTY, but also pick up rates, yen, gold, FX, and selected commodity stress.
- Futures-price correlations recover a classic risk-off map: equity beta is clustered, while dollar strength, duration, and yen are often on the defensive side of ES risk.
- Equity-index ATM implied volatility and ES downside-skew pressure raise forward SPX drawdown event rates.
- Agriculture expands the macro interpretation set, but its top-decile implied-volatility states do not behave like a generic SPX crash warning.
- PCA absorption is useful as a concentration diagnostic, not as a standalone crash predictor.
- The AR/IVM stress overlay improves risk-adjusted backtest behavior in the 2020+ holdout and reduces drawdown in the longer expanding illustration, while VIX term structure remains the tougher direct forecasting benchmark.

The paper should therefore position ChronoSwan as a disciplined stress-regime detection and protection framework rather than a pure return-forecasting model. The novel angle is the combination of event-conditioned PCA, cross-asset option-implied stress, strict point-in-time validation, and a future agent layer that synthesizes macro narrative only after the market-data signal is frozen.


## Remaining Additions

- Add a cleaner continuous-futures return panel from Bloomberg, Nasdaq Data Link, or another licensed source to benchmark realized-return correlations against the AR/IVM selected-maturity futures prices.
- Replace the simple cash de-risking rule with locked hedge-instrument tests: Treasury futures, put spreads, VIX futures, or dynamic beta reduction.
- Compare PCA absorption against dynamic conditional correlation, spillover-network, and absorption-ratio baselines.
- Freeze thresholds/model classes before any further strategy tuning.
- Add timestamp-valid macro/news context only after the structured market-data benchmark is frozen.
